In [ ]:
# jupyter notebook
import nest_asyncio
nest_asyncio.apply()

In [ ]:
from isaacsim.examples.interactive.base_sample import BaseSample
from isaacsim.core.api import World
from isaacsim.robot.manipulators.examples.franka import Franka
from isaacsim.core.utils.nucleus import get_assets_root_path
from isaacsim.core.api.objects import DynamicCuboid
from isaacsim.core.api.robots import Robot

from isaacsim.robot.manipulators.examples.franka.controllers import PickPlaceController

import isaacsim.core.utils.stage as stage_utils

import numpy as np

In [ ]:
class HelloWorld(BaseSample):
    def __init__(self) -> None:
        super().__init__()
        # define assets root path
        self._isaac_assets_path = get_assets_root_path()
        
        # define url of assets
        self.jetbot_url = self._isaac_assets_path + "/Isaac/Robots/NVIDIA/Jetbot/jetbot.usd"
        return

    def setup_scene(self):
        world = self.get_world()
        world.scene.add_default_ground_plane() # https://docs.isaacsim.omniverse.nvidia.com/5.0.0/py/source/extensions/isaacsim.core.api/docs/index.html#isaacsim.core.api.world.World
        stage_utils.add_reference_to_stage(usd_path=self.jetbot_url, prim_path="/World/jetbot")
        jetbot = world.scene.add(Robot(prim_path="/World/jetbot", name="jetbot1", position=np.array([0, -0.3, 0])))         
        franka = world.scene.add(Franka(prim_path="/World/Fancy_Franka", name="fancy_franka"))
        
        world.scene.add(
            DynamicCuboid(
                prim_path="/World/random_cube",
                name="fancy_cube",
                position=np.array([0.3, 0.3, 0.3]),
                scale=np.array([0.0515, 0.0515, 0.0515]),
                color=np.array([0, 0, 1.0]),
            )
        )
        return
    
    # right after the scene setup 
    async def setup_post_load(self):
        self._world = self.get_world()

        # franka
        self._franka = self._world.scene.get_object("fancy_franka")
        self._fancy_cube = self._world.scene.get_object("fancy_cube")
        # initializing a pick and place controller
        self._frankactrl = PickPlaceController(
            name ="pick_place_controller",
            gripper=self._franka.gripper,
            robot_articulation=self._franka,
        )
        self._world.add_physics_callback("franka_step", callback_fn = self.franka_step)
        self._franka.gripper.set_joint_positions(self._franka.gripper.joint_opened_positions)

        # jetbot
        self._jetbot = self._world.scene.get_object("jetbot1")
        self._jetbot_articulation_controller = self._jetbot.get_articulation_controller()
        self._world.add_physics_callback("jetbot_step", callback_fn = self.jetbot_step)
        
        await self._world.play_async()
        return

    async def setup_post_reset():
        #franka
        self._frankactrl.reset()
        self._franka.gripper.set_joint_positions(self._franka.gripper.joint_opened_positions)
        
        await self._world.play_async()
        return

    def franka_step(self, step_size):
        cube_position, _ = self._fancy_cube.get_world_pose()
        goal_position = np.array([-0.3, -0.3, 0.0515 / 2.0])
        current_joint_positions = self._franka.get_joint_positions()
        actions = self._frankactrl.forward(
            picking_position=cube_position,
            placing_position=goal_position,
            current_joint_positions=current_joint_positions,
        )
        self._franka.apply_action(actions)
        if self._frankactrl.is_done():
            self._world.pause()
        return

    def jetbot_step(self, step_size):
        self._jetbot_articulation_controller.apply_action(ArticulationAction(joint_positions=None,
                                                                            joint_efforts=None,
                                                                            joint_velocities= np.array([3,3])
                                                                            #joint_velocities=5 * np.random.rand(2,)
                                                                            ))


In [ ]:
await HelloWorld().load_world_async()